# Lesson 8: Word Embeddings & Semantic Change Detection

In this lesson we explore **distributional word embeddings** — dense vector representations learned
from large text corpora. Words that appear in similar contexts end up close together in vector space.

## Learning Objectives

By the end of this notebook you will be able to:

1. Understand the distributional hypothesis and why it produces useful word representations.
2. Train **Word2Vec** models (CBOW and Skip-gram) on raw text using Gensim.
3. Inspect a word's **nearest neighbours** in vector space.
4. Detect **semantic drift** between two time periods using the Google Books N-gram corpus.
5. Evaluate a word similarity model against the **MEN benchmark** with Spearman rank correlation.

## References

- [**MEN Dataset**](https://www.jair.org/index.php/jair/article/view/10857)
- [**Original MEN files**](https://staff.fnwi.uva.nl/e.bruni/MEN)
- [**Spearman Rank Correlation Explained**](https://www.youtube.com/watch?v=DE58QuNKA-c&t=2s)
- Mikolov et al. (2013). *Efficient Estimation of Word Representations in Vector Space*. arXiv:1301.3781


---
## 1. The Distributional Hypothesis

> *You shall know a word by the company it keeps.* — J.R. Firth (1957)

The **distributional hypothesis** states that words appearing in similar **contexts** tend to have
similar **meanings**. Word2Vec exploits this by training a shallow neural network to predict:

- **CBOW** (Continuous Bag of Words): given the surrounding context words, predict the centre word.
- **Skip-gram** (`sg=1` in Gensim): given the centre word, predict the surrounding context words.

The learned **hidden-layer weights** become the word vectors.

### Key Hyperparameters

| Parameter | Gensim arg | Typical value | Effect |
|-----------|-----------|---------------|--------|
| Vector size | `vector_size` | 100–300 | Embedding dimension |
| Context window | `window` | 3–10 | How many neighbours are considered |
| Architecture | `sg` | 0 (CBOW) / 1 (Skip-gram) | Training objective |
| Min count | `min_count` | 2–5 | Ignore very rare words |
| Epochs | `epochs` | 5–20 | Training iterations over the corpus |


---
## 2. Environment Setup

We import the required libraries and point the working directory to the folder containing
the Google Books N-gram file. **Adjust the paths** if you are running on a different machine.


In [1]:
import os
import re
from random import shuffle
import gzip
import gensim
from gensim.models.word2vec import Word2Vec
from scipy.stats import spearmanr

# ── Path configuration ──────────────────────────────────────────────────────
# Change DATA_DIR to the folder that contains the Google Books n-gram file.
DATA_DIR = "/Users/luca/Data/cattolica/"
MEN_PATH = "/Users/luca/Gits/pfl_2026_unicatt/assets/MEN/MEN_dataset_natural_form_full.txt"

os.chdir(DATA_DIR)
print(f"Working directory: {os.getcwd()}")


Working directory: /Users/luca/Data/cattolica


---
## 3. Loading the Google Books N-gram Corpus

The [Google Books N-gram corpus](https://books.google.com/ngrams) contains billions of word sequences
extracted from millions of books spanning several centuries.

We use the **5-gram file starting with 'ca'** as a manageable sample.
Each line has the format:



We split the data into two sub-corpora based on publication year:

- **`before`**: 1990–1999 — language before the internet era
- **`after`**: 2000–present — modern usage

Part-of-speech tags (e.g. `_NOUN`) are stripped with a regex so the model sees clean tokens.

> **Note**: We read at most 10 million lines to keep RAM usage reasonable.


In [5]:
# Initialise the two corpora as empty lists
sampled = {"before": [], "after": []}

# Regex to strip Penn-Treebank / Google Books POS tags
# e.g. 'running_VERB' becomes 'running'
POS_TAG_RE = re.compile(r"_[A-Z.]+(\s|$|_)")

MAX_LINES = 10_000_000  # safety cap to limit memory use
NGRAM_FILE = "googlebooks-eng-all-5gram-20120701-ca"  # must be in DATA_DIR

c = 0
with open(NGRAM_FILE) as fh:
    for line in fh:
        if c >= MAX_LINES:
            break
        # Each tab-separated field: text, year, match_count, volume_count
        text, year, n1, n2 = line.strip().split("	")
        text = POS_TAG_RE.sub(" ", text)  # strip POS tags
        year = int(year)
        tokens = text.split()
        if year > 1999:
            sampled["after"].append(tokens)
        elif year > 1989:
            sampled["before"].append(tokens)
        c += 1

print(f"Lines read         : {c:,}")
print(f"  'before' sentences: {len(sampled['before']):,}")
print(f"  'after'  sentences: {len(sampled['after']):,}")


TypeError: a bytes-like object is required, not 'str'

---
## 4. Shuffling the Corpora

We shuffle both lists before training.
Word2Vec processes sentences sequentially, so shuffling prevents ordering artefacts
from biasing the embeddings.


In [ ]:
# Shuffle in-place so the sequential order does not influence training
shuffle(sampled["after"])
shuffle(sampled["before"])

corpus_after  = sampled["after"]
corpus_before = sampled["before"]

# Quick sanity-check: print the first 5 tokenised sentences from each split
print("Sample sentences [after]:",  corpus_after[:5])
print("Sample sentences [before]:", corpus_before[:5])


---
## 5. Training Word2Vec Models (CBOW)

We train **two separate** Word2Vec models — one per era — using the default **CBOW** architecture
(`sg=0`).
Training two models on comparable corpora lets us later compare how word neighbourhoods differ
between the two periods — a proxy for **semantic change**.

Gensim's `Word2Vec` accepts an iterable of token lists (sentences).


In [ ]:
# Train on post-2000 text (CBOW, default settings: window=5, vector_size=100)
mafter  = Word2Vec(corpus_after)

# Train on 1990–1999 text
mbefore = Word2Vec(corpus_before)

print("Vocabulary sizes:")
print(f"  mafter  : {len(mafter.wv):,} words")
print(f"  mbefore : {len(mbefore.wv):,} words")


---
## 6. Inspecting Nearest Neighbours

The most intuitive way to evaluate a word embedding model is to look at the **k-nearest neighbours**
of a query word in vector space.

`model.wv.most_similar(word, topn=k)` returns the top-k nearest neighbours
measured by **cosine similarity**:

25953\cos(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \|\mathbf{v}\|}25953

A value of **1** means identical context distributions; **0** means orthogonal (independent).


In [ ]:
# Top-10 nearest neighbours of "king" in the post-2000 model
print("Nearest neighbours of 'king' [after 2000]:")
for word, score in mafter.wv.most_similar("king", topn=10):
    print(f"  {word:<20} {score:.4f}")


In [ ]:
# Same query in the pre-2000 model — compare!
print("Nearest neighbours of 'king' [before 2000]:")
for word, score in mbefore.wv.most_similar("king", topn=10):
    print(f"  {word:<20} {score:.4f}")


---
## 7. Measuring Semantic Change with Spearman Rank Correlation

### 7.1 What is Spearman's ρ?

**Spearman rank correlation** measures how well the *ranking* produced by one variable
agrees with the ranking produced by another, without assuming a linear relationship:

25953\rho = 1 - \frac{6 \sum d_i^2}{n(n^2-1)}25953

where $ is the rank difference for item $.

- ρ = **+1** → perfect rank agreement
- ρ = **0**  → no monotone relationship
- ρ = **−1** → perfectly reversed ranking

### 7.2 Application to Semantic Change

For a given word **w**, we extract its top-20 neighbours from each era's model.
We then compute Spearman's ρ between the two ranked neighbour lists.
A **low ρ** indicates the word's neighbourhood has changed substantially — **semantic drift**.


In [ ]:
def spearman_knn(word, m1, m2, k=20):
    """Compare the top-k neighbours of  across two Word2Vec models.

    Parameters
    ----------
    word : str
        Query word.
    m1, m2 : Word2Vec
        The two models to compare.
    k : int
        Number of nearest neighbours.

    Returns
    -------
    scipy.stats.SpearmanrResult  (statistic = rho, pvalue)
    """
    nbrs1 = [w for w, _ in m1.wv.most_similar(word, topn=k)]
    nbrs2 = [w for w, _ in m2.wv.most_similar(word, topn=k)]
    return spearmanr(nbrs1, nbrs2)


# Example: how stable is the neighbourhood of 'king' across the two eras?
result = spearman_knn("king", mafter, mbefore)
print(f"Spearman rho for 'king': {result.statistic:.4f}  (p={result.pvalue:.4f})")

# Interpretation: a low |rho| suggests the usage of 'king' has drifted


---
## 8. Skip-gram vs CBOW — Does the Architecture Matter?

**Skip-gram** (`sg=1`) inverts the prediction task: given a centre word, predict its context.
It tends to produce better representations for **rare words** and is generally preferred for
semantic analogy tasks.
**CBOW** is faster and often better for frequent words.

We also set `window=3` (smaller context window), which captures **syntactic** rather than
**semantic** relationships more strongly.


In [ ]:
# Skip-gram models with a narrow window (window=3, sg=1)
mafterw2  = Word2Vec(corpus_after,  window=3, sg=1)
mbforew2  = Word2Vec(corpus_before, window=3, sg=1)

print("Skip-gram models trained.")
print(f"  mafterw2  vocab: {len(mafterw2.wv):,}")
print(f"  mbforew2  vocab: {len(mbforew2.wv):,}")


In [ ]:
# Compare KNN for 'king' in the Skip-gram models
print("Skip-gram neighbours of 'king' [after 2000]:")
for word, score in mafterw2.wv.most_similar("king", topn=10):
    print(f"  {word:<20} {score:.4f}")

print()
print("Skip-gram neighbours of 'king' [before 2000]:")
for word, score in mbforew2.wv.most_similar("king", topn=10):
    print(f"  {word:<20} {score:.4f}")


In [ ]:
# Spearman correlation for the Skip-gram models
result_sg = spearman_knn("king", mbforew2, mafterw2)
print(f"Spearman rho (Skip-gram, 'king'): {result_sg.statistic:.4f}  (p={result_sg.pvalue:.4f})")


---
## 9. Evaluating Against the MEN Benchmark

### What is MEN?

The **MEN dataset** (Bruni et al., 2012) contains **3,000 word pairs** each annotated with a
human similarity score on a scale of 0–50.
It covers mostly **concrete, visual concepts** (objects, animals, colours, scenes).

### Evaluation Protocol

For each word pair $ in MEN:
1. Compute the **cosine similarity** according to our model.
2. Collect the **human rating**.
3. Compute **Spearman ρ** between the ranked model scores and ranked human scores.

Higher ρ means the model's similarity judgements better match human intuition.

> Word pairs for which either word is **out-of-vocabulary** (OOV) are skipped.


In [ ]:
# Load the MEN dataset
# Format: word1  word2  score  (space-separated)
men = [line.split() for line in open(MEN_PATH).readlines()]

print(f"MEN pairs loaded: {len(men)}")
print("First 5 rows:", men[:5])


In [ ]:
def evaluate_on_men(model, men_pairs):
    """Compute Spearman rho between model cosine similarities and MEN human ratings.

    Parameters
    ----------
    model : Word2Vec
    men_pairs : list of [word1, word2, score_str]

    Returns
    -------
    tuple: (SpearmanrResult, n_covered)
    """
    model_scores = []
    human_scores = []
    for w1, w2, val in men_pairs:
        try:
            model_scores.append(model.wv.similarity(w1, w2))
            human_scores.append(float(val))
        except KeyError:
            pass  # skip OOV pairs
    return spearmanr(model_scores, human_scores), len(model_scores)


rho_after,  n_after  = evaluate_on_men(mafter,  men)
rho_before, n_before = evaluate_on_men(mbefore, men)

print(f"CBOW [after  2000]: rho = {rho_after.statistic:.4f}  (coverage: {n_after}/{len(men)})")
print(f"CBOW [before 2000]: rho = {rho_before.statistic:.4f}  (coverage: {n_before}/{len(men)})")


In [ ]:
# Evaluate the Skip-gram models on MEN
rho_after_sg,  n_after_sg  = evaluate_on_men(mafterw2,  men)
rho_before_sg, n_before_sg = evaluate_on_men(mbforew2, men)

print(f"Skip-gram [after  2000]: rho = {rho_after_sg.statistic:.4f}  (coverage: {n_after_sg}/{len(men)})")
print(f"Skip-gram [before 2000]: rho = {rho_before_sg.statistic:.4f}  (coverage: {n_before_sg}/{len(men)})")


---
## 10. Results Summary

| Model | Era | Architecture | Window | Spearman ρ (MEN) |
|-------|-----|--------------|--------|------------------|
| mafter   | post-2000  | CBOW      | 5 (default) | *(fill in above)* |
| mbefore  | pre-2000   | CBOW      | 5 (default) | *(fill in above)* |
| mafterw2 | post-2000  | Skip-gram | 3           | *(fill in above)* |
| mbforew2 | pre-2000   | Skip-gram | 3           | *(fill in above)* |

### Questions to consider

- Does the **era** (before/after 2000) affect the similarity scores?
- Does the **architecture** (CBOW vs Skip-gram) matter more than the window size?
- MEN focuses on **visual/concrete** concepts — do distributional models capture this well?


---
## Exercises

Complete the following tasks.
Aim for **clear, well-commented code**.
Write your discussion in the markdown cells provided.


### Exercise 1 — Exploring Word Analogies

Word2Vec supports **analogy arithmetic** via
`model.wv.most_similar(positive=[...], negative=[...])`.

1. Try the classic analogy: king − man + woman (expected answer: queen).
2. Try three more analogies of your choice (e.g. country–capital, profession–gender).
3. Do the two eras (`mafter` and `mbefore`) give different answers? Why?

**Hint**: `model.wv.most_similar(positive=['king', 'woman'], negative=['man'], topn=5)`


In [ ]:
# Your code here
# ── Analogy: king - man + woman ─────────────────────────────────────────────


*Your observations (edit this cell):*

- ...


### Exercise 2 — Hyperparameter Sensitivity

Train four new CBOW models on `corpus_after`, varying one hyperparameter at a time:

| Model | `vector_size` | `window` |
|-------|--------------|----------|
| A | 50  | 5 |
| B | 300 | 5 |
| C | 100 | 2 |
| D | 100 | 10 |

For each model:
- Report the Spearman ρ on MEN.
- Print the top-5 neighbours of `'computer'` and `'music'`.

Discuss: which configuration performs best, and why?


In [ ]:
# Your code here
# ── Train Model A (vector_size=50, window=5) ────────────────────────────────


*Your observations (edit this cell):*

- ...


### Exercise 3 — Semantic Drift Analysis

Using `spearman_knn`, compute the KNN-Spearman correlation between `mbefore` and `mafter`
for the following words:



1. Sort the words from **most drifted** (lowest |ρ|) to **least drifted**.
2. For the top-2 most drifted words, print the top-10 neighbours in each era
   and explain the semantic shift you observe.

> Some words may be OOV in one model — handle the `KeyError` gracefully.


In [ ]:
# Your code here
words_to_test = ['internet', 'mobile', 'bank', 'virus', 'king', 'cloud', 'network']

# ── Compute drift scores ─────────────────────────────────────────────────────


*Your observations (edit this cell):*

- ...


### Exercise 4 — MEN Subset Analysis

The MEN dataset contains word pairs with scores ranging from 0 (no similarity) to 50 (identical).

1. Split MEN into **three bins**:
   - `low`:  score <= 16
   - `mid`:  16 < score <= 33
   - `high`: score > 33
2. Evaluate `mafter` separately on each bin.
3. In which range does the model perform best?
   Discuss why distributional models might struggle with very low or very high similarity pairs.


In [ ]:
# Your code here
# ── Split MEN into bins ──────────────────────────────────────────────────────


*Your observations (edit this cell):*

- ...


### Exercise 5 — Visualising Word Clusters (Bonus)

Use **PCA** or **t-SNE** to reduce the word vectors to 2D and visualise a selected vocabulary.

1. Choose 50–80 words covering at least 4 semantic categories (e.g. animals, countries, food, motion verbs).
2. Colour the points by category.
3. Do the clusters in `mafter` look different from those in `mbefore`?

**Packages**: `sklearn.decomposition.PCA`, `sklearn.manifold.TSNE`, `matplotlib`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
# from sklearn.manifold import TSNE  # uncomment for t-SNE

# ── Define vocabulary groups ─────────────────────────────────────────────────
groups = {
    "animals":   ["dog", "cat", "horse", "lion", "tiger", "bird", "fish"],
    "countries": ["france", "germany", "italy", "spain", "china", "canada"],
    "food":      ["bread", "cheese", "rice", "pasta", "soup", "cake"],
    "verbs":     ["run", "jump", "walk", "swim", "fly", "drive"],
}

# Filter to words actually present in the model vocabulary
filtered = {cat: [w for w in wlist if w in mafter.wv] for cat, wlist in groups.items()}

# TODO: extract vectors, run PCA, plot with coloured labels


*Your observations (edit this cell):*

- ...


# Spearman's Rank Correlation from scratch
$r = 1 - \frac{6\sum{d^2}}{n^3-n}$


In [150]:
distance = [50, 175, 250, 375, 425, 585, 720, 810, 875, 950]
price = [1.80, 1.25, 2, 1, 1.10, 1.20, 0.8, 0.60, 1.05, 0.85]


spearmanr(distance, price)

SignificanceResult(statistic=-0.7575757575757575, pvalue=0.011143446799694208)

In [161]:
def our_spearman(x,y):
    ## The formula states that for every pair of observations we need to compute
    ## their rank in their distribution.
    ranked_x = rank_observations(x)
    ranked_y = rank_observations(y)
    d_squared = [(i-j)**2 for i, j in zip(ranked_x, ranked_y)]
    numerator = 6*sum(d_squared)
    denominator = (len(x))**3 -len(x)
    return numerator, denominator

    return None
def rank_observations(x):
    ranked = sorted(x, reverse=True)
    return [ranked.index(i) +1 for i in x]

our_spearman(distance,price)

(1740, 990)